In [ ]:
import json
from pathlib import Path

In [ ]:

# Replace with your actual downloaded filename
file_path = Path("../data/MasterPlan2025PlanningAreaBoundaryNoSea.geojson")

with open(file_path, "r", encoding="utf-8") as f:
    ura_data = json.load(f)

print("GeoJSON type:", ura_data.get("type"))
print("Number of features:", len(ura_data.get("features", [])))

first_feature = ura_data["features"][0]

print("\nAvailable properties:")
print(list(first_feature["properties"].keys()))

print("\nGeometry type:")
print(first_feature["geometry"]["type"])

print("\nFirst coordinate sample:")
print(str(first_feature["geometry"]["coordinates"])[:200])

print("\nCoordinate reference system:")
print(ura_data.get("crs", "Not explicitly specified"))

In [ ]:
%pip install shapely

In [ ]:
import pandas as pd
import json
from shapely.geometry import shape

In [ ]:
# Uses ura_data already loaded in your previous cell

records = []

for feature in ura_data["features"]:
    record = feature["properties"].copy()

    # Preserve complete geographic boundary
    record["geometry_wkt"] = shape(feature["geometry"]).wkt

    records.append(record)

ura_df = pd.DataFrame(records)

# Check that no attributes were dropped
expected_columns = set(ura_data["features"][0]["properties"].keys())
actual_columns = set(ura_df.columns)

assert expected_columns.issubset(actual_columns)
assert len(ura_df) == len(ura_data["features"])

print("Rows:", len(ura_df))
print("Columns:", len(ura_df.columns))
print("All original attributes preserved:",
      expected_columns.issubset(actual_columns))

ura_df.head()

In [ ]:
ura_df = ura_df.rename(columns={
    "SHAPE.AREA": "SHAPE_AREA",
    "SHAPE.LEN": "SHAPE_LEN"
})

print(ura_df.columns.tolist())

In [ ]:
output_file = file_path.parent / "ura_planning_areas_2025.csv"
ura_df.to_csv(output_file, index=False)

print("Saved to:", output_file)